# Phase 2: V1/V2 Format CHIVA Training
## Two-Phase Architecture: Phase 1 (Medical Pretraining) → Phase 2 (V1/V2 Task-Specific)
### Qwen2.5-7B with Phase 1 merged + Fresh Phase 2 LoRA

**Critical:** This notebook trains on V1 (clip notation) and V2 (medical terminology) formats
that match ubuntu_evaluation.py query structures. Model will output proper JSON with confidence scores.

## Cell 1: Setup, Load Phase 1, Apply Phase 2 LoRA

In [ ]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

import torch
import json
import jsonlines
from pathlib import Path
from datetime import datetime
from torch.utils.data import Dataset, DataLoader
import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, get_peft_model, LoraConfig, TaskType

print("✓ Imports successful")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ============================================================
# CONFIGURATION - CHANGE THESE PATHS
# ============================================================

BASE_MODEL = "Qwen/Qwen2.5-7B"
PHASE1_LORA = "/home/username/llm_finetuning/qwen_medical_lora_gpu"  # CHANGE THIS
BASE_DIR = "/home/username/llm_finetuning"  # CHANGE THIS

DATA_DIR = f"{BASE_DIR}/latest_data"
DATASET_FILE = f"{DATA_DIR}/training_data_comprehensive.jsonl"
OUTPUT_DIR = f"{BASE_DIR}/qwen_chiva_v1_v2_lora"
CACHE_DIR = f"{BASE_DIR}/.cache"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

# Verify dataset exists
if not os.path.exists(DATASET_FILE):
    print(f"ERROR: {DATASET_FILE} not found!")
else:
    with jsonlines.open(DATASET_FILE) as f:
        count = sum(1 for _ in f)
    print(f"\nDataset found: {count} examples (V1 + V2 formats with comprehensive NO SHUNT cases)")
    print(f"Output dir: {OUTPUT_DIR}")

# ============================================================
# LOAD BASE MODEL
# ============================================================

print("\n[1/5] Loading base model (this takes ~2 min)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    cache_dir=CACHE_DIR,
)
print(f"Base model loaded - {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B parameters")

# ============================================================
# LOAD AND MERGE PHASE 1 LORA
# ============================================================

print("\n[2/5] Loading Phase 1 LoRA adapter...")
try:
    model = PeftModel.from_pretrained(model, PHASE1_LORA)
    print(f"Phase 1 LoRA loaded from: {PHASE1_LORA}")

    print("\n[3/5] Merging Phase 1 LoRA into base model...")
    model = model.merge_and_unload()
    print(f"Phase 1 merged (medical knowledge baked into weights)")
    torch.cuda.empty_cache()
except Exception as e:
    print(f"Warning: Could not load Phase 1 LoRA: {e}")
    print(f"Proceeding with base model only")

# ============================================================
# APPLY PHASE 2 LORA
# ============================================================

print("\n[4/5] Applying Phase 2 LoRA for task-specific training...")

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.train()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Phase 2 LoRA applied")
print(f"  Trainable: {trainable / 1e6:.1f}M ({100 * trainable / total:.2f}%)")
print(f"  Total: {total / 1e9:.1f}B")

# ============================================================
# DATASET CLASS
# ============================================================

print("\n[5/5] Setting up dataset...")

class CHIVADataset(Dataset):
    """CHIVA training data with V1/V2 formats and JSON outputs."""

    def __init__(self, jsonl_file, tokenizer, max_len=512):
        self.data = []
        with jsonlines.open(jsonl_file) as f:
            for line in f:
                self.data.append(line)
        self.tokenizer = tokenizer
        self.max_len = max_len
        print(f"Loaded {len(self.data)} examples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # Format: Input query + JSON output
        text = f"{item['input']}\n{item['output']}"

        enc = self.tokenizer(
            text,
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )

        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
        }

# ============================================================
# LOAD TOKENIZER
# ============================================================

print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    cache_dir=CACHE_DIR,
)
tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer loaded - Vocab: {len(tokenizer)}")

print("\n" + "="*80)
print("READY FOR TRAINING")
print("="*80)
print(f"Phase 1: Merged into base model")
print(f"Phase 2: Fresh LoRA (trainable)")
print(f"Dataset: V1 (clip notation) + V2 (medical terminology) + Extensive NO SHUNT")
print(f"Examples: {count} (30 per shunt type × 6 + 36 NO SHUNT)")
print("="*80)

## Cell 2: Prepare Dataset & Optimizer

In [ ]:
print("Loading dataset...")
dataset = CHIVADataset(DATASET_FILE, tokenizer, max_len=512)

batch_size = 1
num_epochs = 10

dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"\nDataset ready")
print(f"  Examples: {len(dataset)}")
print(f"  Batch size: {batch_size}")
print(f"  Total batches: {len(dataloader)}")
print(f"  Gradient accumulation: 2 steps")
print(f"  Effective batch: {batch_size * 2}")

# ============================================================
# OPTIMIZER & SCHEDULER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=0.01,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

print(f"\nOptimizer configured")
print(f"  Learning rate: 5e-5")
print(f"  Epochs: {num_epochs}")
print(f"  Scheduler: Cosine Annealing")

## Cell 3: Training Loop (8-12 minutes on GPU)

In [ ]:
print("="*80)
print("STARTING PHASE 2 TRAINING (V1/V2 FORMAT)")
print("="*80)
print(f"Start: {datetime.now().isoformat()}\n")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}\n")
model = model.to(device)

total_loss = 0
global_step = 0
losses_per_epoch = []

for epoch in range(num_epochs):
    epoch_loss = 0
    epoch_steps = 0

    print(f"Epoch {epoch+1}/{num_epochs}")
    print("-" * 40)

    for batch_idx, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=input_ids
        )
        loss = outputs.loss

        # Backward
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # Step (every 2 batches for gradient accumulation)
        if (batch_idx + 1) % 2 == 0:
            optimizer.step()
            optimizer.zero_grad()

        epoch_loss += loss.item()
        total_loss += loss.item()
        epoch_steps += 1
        global_step += 1

        if (batch_idx + 1) % 10 == 0:
            avg = epoch_loss / epoch_steps
            print(f"  Batch {batch_idx+1}/{len(dataloader)}, Loss: {loss.item():.4f}, Avg: {avg:.4f}")

    scheduler.step()
    avg_epoch = epoch_loss / epoch_steps
    losses_per_epoch.append(avg_epoch)
    print(f"[OK] Epoch {epoch+1} - Loss: {avg_epoch:.4f}\n")

print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80)
print(f"End: {datetime.now().isoformat()}")
print(f"Final loss: {total_loss / global_step:.4f}")
print(f"Total steps: {global_step}")

## Cell 4: Save Model & Test

In [ ]:
# ============================================================
# SAVE MODEL
# ============================================================

print("\nSaving model...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

config = {
    'model': BASE_MODEL,
    'phase1_merged': True,
    'num_examples': len(dataset),
    'epochs': num_epochs,
    'learning_rate': 5e-5,
    'final_loss': total_loss / global_step,
    'total_steps': global_step,
    'dataset_format': 'V1 (clip notation) + V2 (medical terminology)',
    'timestamp': datetime.now().isoformat(),
}

with open(os.path.join(OUTPUT_DIR, 'training_config.json'), 'w') as f:
    json.dump(config, f, indent=2)

print(f"[OK] Model saved to: {OUTPUT_DIR}")

# ============================================================
# PLOT TRAINING LOSS
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(losses_per_epoch) + 1), losses_per_epoch, marker='o', linestyle='-', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Average Loss', fontsize=12)
plt.title('V1/V2 CHIVA Training Loss', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_loss.png", dpi=100)
plt.show()

print(f"[OK] Loss plot saved")

# ============================================================
# TEST THE MODEL
# ============================================================

print("\n" + "="*80)
print("TESTING FINE-TUNED MODEL")
print("="*80 + "\n")

# V1 format test
v1_test = """Classify the shunt types given these clips:
  Clip 00: EP N1->N2  y=0.050 [SFJ-ENTRY=INCOMPETENT]
  Clip 01: RP N2->N1  y=0.132 [GSV-TRUNK-REFLUX: N2->N1]

Provide response as JSON with shunt_type, confidence, and reasoning."""

# V2 format test
v2_test = """Based on the following duplex findings:
Duplex ultrasound demonstrates: antegrade flow from deep femoral vein to saphenous trunk indicating saphenofemoral junction incompetence; retrograde reflux within saphenous trunk toward deep system; in patient with chronic venous insufficiency.

Determine the CHIVA shunt type. Provide response as JSON with shunt_type, confidence, and reasoning."""

test_cases = [
    ("V1 Format (Clip Notation)", v1_test),
    ("V2 Format (Medical Terminology)", v2_test),
]

model.eval()
with torch.no_grad():
    for test_name, prompt in test_cases:
        print(f"[{test_name}]")
        print(f"Input:\n{prompt}\n")

        inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)

        outputs = model.generate(
            inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
        )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Extract just the generated part
        response = response[len(prompt):].strip()

        print(f"Output:\n{response}\n")
        print("-" * 80 + "\n")

print("[OK] Testing complete")

# ============================================================
# SUMMARY
# ============================================================

print("\n" + "="*80)
print("PHASE 2 TRAINING COMPLETE")
print("="*80)
print(f"\nModel Location: {OUTPUT_DIR}")
print(f"\nTraining Summary:")
print(f"  Phase 1: Medical pretraining (merged into base)")
print(f"  Phase 2: V1/V2 task-specific fine-tuning")
print(f"  Examples: {len(dataset)} (V1 clip notation + V2 medical terminology)")
print(f"  Epochs: {num_epochs}")
print(f"  Final loss: {total_loss / global_step:.4f}")
print(f"  Total steps: {global_step}")
print(f"\n[CRITICAL] This model understands:")
print(f"  1. V1 format queries (clip notation with anatomical labels)")
print(f"  2. V2 format queries (medical terminology descriptions)")
print(f"  3. Outputs proper JSON with confidence and reasoning")
print(f"  4. Generalizes patterns from real patient data")
print("\n" + "="*80)